# FDAX Local Pivot Point Conditional Probabilities
## First Hour (9:00-10:00) Derived Pivot Points - Sequential Analysis

**Data Source:** FDAX M5 OHLCV from Databento (local CSV), ~220 trading days (Mar 2025 - Jan 2026)

**Core Difference from Standard Pivots:**
- **Standard Pivots**: Calculated from PREVIOUS day's High/Low/Close
- **Local Pivots (L)**: Calculated from CURRENT day's FIRST HOUR (9:00-10:00) High/Low/Close

**Methodology:**
1. **Calculate First Hour Range (9:00-10:00 Berlin time):**
   - Use M5 candles from 9:00-9:55 (12 bars)
   - First Hour High = max(high) across all 12 bars
   - First Hour Low = min(low) across all 12 bars
   - First Hour Close = 9:55 candle close

2. **Derive Local Pivot Points (Standard Formulas):**
   - LPP = (H + L + C) / 3
   - LR1 = (2 x LPP) - L
   - LS1 = (2 x LPP) - H
   - LR2 = LPP + (H - L)
   - LS2 = LPP - (H - L)
   - LR3 = H + 2 x (LPP - L)
   - LS3 = L - 2 x (H - LPP)

3. **Classify Opening Zone at 10:00:**
   - Where is the 10:00 price relative to local pivots?
   - Zones: Above_LR3, LR2_LR3, LR1_LR2, LPP_LR1, LS1_LPP, LS2_LS1, LS3_LS2, Below_LS3

4. **Calculate Sequential Conditional Probabilities:**
   - P(Target | Condition, Zone) for 10:00-17:30 session
   - Track FIRST timestamp when each level touched
   - Only count if timestamp_target >= timestamp_condition (allows same-bar touches)
   - **NO TEMPORAL BIAS**

**Target Coverage (6 up + 6 down from condition):**
```
Example: Zone LPP_LR1, Condition "reached LR1"
  Upward (6 targets):
    LR1_LR2_025, LR1_LR2_050, LR1_LR2_075, LR2, LR2_LR3_025, LR2_LR3_050
  
  Downward (6 targets):
    LPP_LR1_075, LPP_LR1_050, LPP_LR1_025, LPP, LS1_LPP_075, LS1_LPP_050
```

**Hypothesis:**
- First hour establishes intraday support/resistance levels
- These "local" pivots may be more relevant than standard pivots for same-day trading
- Conditional probabilities after hitting local levels reveal intraday mean-reversion vs momentum

**Sample Size Note:** ~220 trading days (smaller than DE40 study which had ~1276 days).
Results with N < 30 should be treated with caution.

**Data:** FDAX M5 OHLCV, Mar 2025 - Jan 2026, RTH (09:00-17:30 Berlin)

---

## Step 1: Setup and Data Loading

In [ ]:
import sys
sys.path.insert(0, '../../')

from shared.data_module import load_from_csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (22, 16)

print('[OK] Dependencies loaded')
print('='*80)

## Step 2: Load FDAX M5 Data from CSV

In [ ]:
print('\n[STEP 2] Load FDAX M5 Data from CSV')
print('='*80)

# Load via our new CSV loader - handles mixed CET/CEST offsets through UTC
df_m5 = load_from_csv('shared/FDAX_M5_OHLCV.csv', timezone='Europe/Berlin')

print(f'[OK] Loaded {len(df_m5)} M5 candles')
print(f'Date range: {df_m5.index.min()} to {df_m5.index.max()}')
print(f'Timezone: {df_m5.index.tz}')

# Extract time components
df_m5['date'] = df_m5.index.date
df_m5['hour'] = df_m5.index.hour
df_m5['minute'] = df_m5.index.minute
df_m5['time'] = df_m5.index.time

# Filter to RTH (09:00-17:30)
df_m5_rth = df_m5[
    (df_m5['hour'] >= 9) & 
    ((df_m5['hour'] < 17) | ((df_m5['hour'] == 17) & (df_m5['minute'] <= 30)))
].copy()

n_days = df_m5_rth['date'].nunique()
print(f'\n[OK] RTH filtered (09:00-17:30): {len(df_m5_rth)} candles')
print(f'Trading days: {n_days}')
print(f'Date range: {df_m5_rth.index.min().date()} to {df_m5_rth.index.max().date()}')
print(f'\n[NOTE] Data from Databento FDAX futures (local CSV, no DB required)')

## Step 3: Calculate Local Pivot Points from First Hour (9:00-10:00)

**Standard Pivot Formulas Applied to First Hour:**
- H = High of 9:00-10:00 range (max high across M5 bars)
- L = Low of 9:00-10:00 range (min low across M5 bars)
- C = Close of 9:00-10:00 range (9:55 close)
- LPP = (H + L + C) / 3
- LR1 = (2 x LPP) - L
- LS1 = (2 x LPP) - H
- LR2 = LPP + (H - L)
- LS2 = LPP - (H - L)
- LR3 = H + 2 x (LPP - L)
- LS3 = L - 2 x (H - LPP)

In [ ]:
print('\n[STEP 3] Calculate Local Pivot Points (First Hour 9:00-10:00)')
print('='*80)

# Extract first hour data (9:00-9:55)
df_first_hour = df_m5_rth[
    (df_m5_rth['hour'] == 9) & (df_m5_rth['minute'] < 60)
].copy()

print(f'[OK] Extracted first hour candles (9:00-9:55): {len(df_first_hour)} bars')

# Aggregate by day
daily_data = []

for date, day_bars in df_first_hour.groupby('date'):
    if len(day_bars) < 10:  # Need at least 10 bars (50 mins)
        continue
    
    # First hour H/L/C
    H = day_bars['high'].max()
    L = day_bars['low'].min()
    C = day_bars.iloc[-1]['close']
    
    # Calculate Local Pivot Point using STANDARD formulas
    LPP = (H + L + C) / 3
    
    LR1 = (2 * LPP) - L
    LS1 = (2 * LPP) - H
    
    LR2 = LPP + (H - L)
    LS2 = LPP - (H - L)
    
    LR3 = H + 2 * (LPP - L)
    LS3 = L - 2 * (H - LPP)
    
    daily_data.append({
        'date': date,
        'first_hour_high': H,
        'first_hour_low': L,
        'first_hour_close': C,
        'LPP': LPP,
        'LR1': LR1,
        'LS1': LS1,
        'LR2': LR2,
        'LS2': LS2,
        'LR3': LR3,
        'LS3': LS3,
    })

df_daily = pd.DataFrame(daily_data)

print(f'[OK] Calculated local pivots for {len(df_daily)} trading days')
print(f'\nSample (first 5 days):')
print(df_daily[['date', 'first_hour_high', 'first_hour_low', 'first_hour_close', 'LPP', 'LR1', 'LS1']].head())
print(f'\n[NOTE] Standard pivot formulas applied to first hour (9:00-9:55) High/Low/Close')

## Step 4: Get 10:00 Price and Classify Opening Zone

In [ ]:
print('\n[STEP 4] Get 10:00 Price and Classify Opening Zone')
print('='*80)

# Extract 10:00 price (first bar at 10:00)
df_10am = df_m5_rth[
    (df_m5_rth['hour'] == 10) & (df_m5_rth['minute'] == 0)
][['date', 'open', 'high', 'low', 'close']].copy()

df_10am = df_10am.rename(columns={
    'open': 'price_10am',
    'high': 'high_10am',
    'low': 'low_10am',
    'close': 'close_10am'
})

# Merge with daily data
df_daily = df_daily.merge(df_10am, on='date', how='left')

# Drop days without 10:00 data
df_daily = df_daily[df_daily['price_10am'].notna()].reset_index(drop=True)

print(f'[OK] Merged 10:00 prices: {len(df_daily)} valid days')

# Classify opening zone at 10:00
def classify_local_zone(row):
    """Classify where 10:00 price is relative to local pivots."""
    price = row['price_10am']
    
    if price > row['LR3']:
        return 'Above_LR3'
    elif row['LR2'] < price <= row['LR3']:
        return 'LR2_LR3'
    elif row['LR1'] < price <= row['LR2']:
        return 'LR1_LR2'
    elif row['LPP'] < price <= row['LR1']:
        return 'LPP_LR1'
    elif row['LS1'] < price <= row['LPP']:
        return 'LS1_LPP'
    elif row['LS2'] < price <= row['LS1']:
        return 'LS2_LS1'
    elif row['LS3'] < price <= row['LS2']:
        return 'LS3_LS2'
    else:
        return 'Below_LS3'

df_daily['opening_zone'] = df_daily.apply(classify_local_zone, axis=1)

print(f'[OK] Classified opening zones at 10:00')
print(f'\nZone distribution:')
for zone, count in df_daily['opening_zone'].value_counts().items():
    pct = count / len(df_daily) * 100
    print(f'  {zone:12} {count:4d} days ({pct:.1f}%)')

print(f'\n[NOTE] With ~{len(df_daily)} days, zones with < 30 days have low statistical power')

## Step 5: Define Local Target Levels (Extended Fractional Targets)

In [ ]:
print('\n[STEP 5] Define Local Target Levels (Extended)')
print('='*80)

def calculate_local_targets(row):
    """
    Calculate all local fractional targets.
    All prefixed with 'L' for Local.
    """
    targets = {}
    
    # Major levels
    targets['LS3'] = row['LS3']
    targets['LS2'] = row['LS2']
    targets['LS1'] = row['LS1']
    targets['LPP'] = row['LPP']
    targets['LR1'] = row['LR1']
    targets['LR2'] = row['LR2']
    targets['LR3'] = row['LR3']
    
    # LS2-LS3 range
    dist = row['LS2'] - row['LS3']
    targets['LS2_LS3_025'] = row['LS3'] + 0.25 * dist
    targets['LS2_LS3_050'] = row['LS3'] + 0.50 * dist
    targets['LS2_LS3_075'] = row['LS3'] + 0.75 * dist
    
    # LS1-LS2 range
    dist = row['LS1'] - row['LS2']
    targets['LS1_LS2_025'] = row['LS2'] + 0.25 * dist
    targets['LS1_LS2_050'] = row['LS2'] + 0.50 * dist
    targets['LS1_LS2_075'] = row['LS2'] + 0.75 * dist
    
    # LS1-LPP range
    dist = row['LPP'] - row['LS1']
    targets['LS1_LPP_025'] = row['LS1'] + 0.25 * dist
    targets['LS1_LPP_050'] = row['LS1'] + 0.50 * dist
    targets['LS1_LPP_075'] = row['LS1'] + 0.75 * dist
    
    # LPP-LR1 range
    dist = row['LR1'] - row['LPP']
    targets['LPP_LR1_025'] = row['LPP'] + 0.25 * dist
    targets['LPP_LR1_050'] = row['LPP'] + 0.50 * dist
    targets['LPP_LR1_075'] = row['LPP'] + 0.75 * dist
    
    # LR1-LR2 range
    dist = row['LR2'] - row['LR1']
    targets['LR1_LR2_025'] = row['LR1'] + 0.25 * dist
    targets['LR1_LR2_050'] = row['LR1'] + 0.50 * dist
    targets['LR1_LR2_075'] = row['LR1'] + 0.75 * dist
    
    # LR2-LR3 range
    dist = row['LR3'] - row['LR2']
    targets['LR2_LR3_025'] = row['LR2'] + 0.25 * dist
    targets['LR2_LR3_050'] = row['LR2'] + 0.50 * dist
    targets['LR2_LR3_075'] = row['LR2'] + 0.75 * dist
    
    return targets

df_daily['targets'] = df_daily.apply(calculate_local_targets, axis=1)

print('[OK] Local target levels calculated')
print(f'\nTotal targets per day: {len(df_daily.iloc[0]["targets"])} levels')
print('\nExample targets (first day):')
example = df_daily.iloc[0]['targets']
for key in ['LS2', 'LS1', 'LS1_LPP_050', 'LPP', 'LPP_LR1_050', 'LR1', 'LR2']:
    print(f'  {key:15} = {example[key]:.2f}')

## Step 6: Track FIRST Touch Timestamps (10:00-17:30 Session)

**Critical:** We only analyze 10:00-17:30 (after local pivots are established).
This eliminates any temporal bias - pivots are calculated from the first hour,
then we only measure what happens AFTER that.

For each level, we record the FIRST M5 bar where the bar's [low, high] range
contains the target price. This gives us the earliest possible touch timestamp.

In [ ]:
print('\n[STEP 6] Track First Touch Timestamps (10:00-17:30 Session)')
print('='*80)
print('\n[WARNING] Processing intrabar data - may take 1-2 minutes...')

# Filter M5 data to 10:00-17:30 only
df_m5_session = df_m5_rth[
    (df_m5_rth['hour'] >= 10)
].copy()

print(f'[OK] Session data filtered: {len(df_m5_session)} M5 bars (10:00-17:30)')

def track_first_touches_session(date, targets_dict, df_m5_day):
    """
    Track FIRST timestamp when each local target is touched.
    Only for 10:00-17:30 session.
    """
    first_touches = {key: None for key in targets_dict.keys()}
    
    # Loop chronologically through M5 bars
    for timestamp, bar in df_m5_day.iterrows():
        bar_high = bar['high']
        bar_low = bar['low']
        
        for target_name, target_price in targets_dict.items():
            if first_touches[target_name] is not None:
                continue
            
            if bar_low <= target_price <= bar_high:
                first_touches[target_name] = timestamp
    
    return first_touches

# Process each day
first_touch_data = []

for idx, day_row in df_daily.iterrows():
    date = day_row['date']
    targets_dict = day_row['targets']
    
    # Get session M5 bars (10:00-17:30)
    df_m5_day = df_m5_session[df_m5_session['date'] == date].copy()
    
    if len(df_m5_day) < 10:
        continue
    
    first_touches = track_first_touches_session(date, targets_dict, df_m5_day)
    
    first_touch_data.append({
        'date': date,
        'first_touches': first_touches
    })
    
    if (idx + 1) % 50 == 0:
        print(f'  Processed {idx + 1}/{len(df_daily)} days...')

df_touch = pd.DataFrame(first_touch_data)
df_daily = df_daily.merge(df_touch, on='date', how='left')

print(f'\n[OK] First touch timestamps tracked for {len(df_daily)} days')
print('[CONFIRMED] Temporal bias eliminated - tracking first touch chronologically')
print('[CONFIRMED] Only 10:00-17:30 session analyzed (after pivots established)')

## Step 7: Calculate Sequential Conditional Probabilities

**Logic:** P(Target | Condition, Zone) where timestamp_target >= timestamp_condition

The `>=` (not strict `>`) allows legitimate same-bar touches. This is correct because
if a single M5 bar's range covers both the condition level AND the target level,
then both were indeed touched during that bar - we should count it.

This was a critical fix from the DE40 study - using strict `>` was too restrictive
and missed legitimate touches that happened within the same 5-minute candle.

In [ ]:
print('\n[STEP 7] Calculate Sequential Conditional Probabilities')
print('='*80)


def calculate_conditional_probs_local(df, zone_name, condition_level):
    """
    Calculate P(Target | Condition, Zone) with temporal ordering.
    target_ts >= condition_ts (allows same-bar touches)
    """
    zone_data = df[df['opening_zone'] == zone_name].copy()
    
    if len(zone_data) < 10:
        return None
    
    # Filter to days where condition was reached
    zone_data['condition_timestamp'] = zone_data['first_touches'].apply(
        lambda ft: ft.get(condition_level) if ft is not None else None
    )
    
    condition_met = zone_data[zone_data['condition_timestamp'].notna()].copy()
    n_condition = len(condition_met)
    
    if n_condition < 5:
        return None
    
    target_keys = list(condition_met.iloc[0]['targets'].keys())
    
    results = []
    for target_key in target_keys:
        if target_key == condition_level:  # Skip self-reference
            results.append({
                'target': target_key,
                'count_sequential': 0,
                'prob_conditional': 1.0,
                'prob_unconditional': 0,
                'prob_delta': 0,
            })
            continue
            
        n_sequential = 0
        n_total_target = 0
        
        for _, row in condition_met.iterrows():
            condition_ts = row['condition_timestamp']
            target_ts = row['first_touches'].get(target_key) if row['first_touches'] is not None else None
            
            # >= allows same-bar touches (critical fix from DE40 study)
            if target_ts is not None and condition_ts is not None and target_ts >= condition_ts:
                n_sequential += 1
        
        for _, row in zone_data.iterrows():
            if row['first_touches'] is not None:
                target_ts = row['first_touches'].get(target_key)
                if target_ts is not None:
                    n_total_target += 1
        
        prob_conditional = n_sequential / n_condition if n_condition > 0 else 0
        prob_unconditional = n_total_target / len(zone_data) if len(zone_data) > 0 else 0
        
        results.append({
            'target': target_key,
            'count_sequential': n_sequential,
            'prob_conditional': prob_conditional,
            'prob_unconditional': prob_unconditional,
            'prob_delta': prob_conditional - prob_unconditional,
        })
    
    return pd.DataFrame(results)


print('[OK] Sequential conditional probability function ready')
print('[CONFIRMED] >= condition_ts allows legitimate same-bar touches')

## Step 8: Analyze Key Zones (6 Up + 6 Down from Condition)

We analyze the two most common zones (LPP_LR1 and LS1_LPP) which typically
capture the majority of days. With ~220 total days, outer zones will have
too few samples for reliable statistics.

For each zone, we test several condition levels and measure:
- **Conditional probability**: P(Target | Condition reached)
- **Unconditional probability**: P(Target) regardless of condition
- **Delta**: Conditional - Unconditional (positive = condition adds information)

In [ ]:
print('\n[STEP 8] Analyze Key Zones with Extended Target Coverage')
print('='*80)

# Define conditions for each zone
zone_conditions = {
    'LS1_LPP': ['LS2', 'LS1_LS2_050', 'LS1', 'LS1_LPP_025', 'LS1_LPP_050', 'LS1_LPP_075', 'LPP', 'LPP_LR1_050', 'LR1'],
    'LPP_LR1': ['LS1', 'LS1_LPP_050', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075', 'LR1', 'LR1_LR2_050', 'LR2'],
    'LR1_LR2': ['LPP', 'LPP_LR1_050', 'LR1', 'LR1_LR2_025', 'LR1_LR2_050', 'LR1_LR2_075', 'LR2', 'LR2_LR3_050', 'LR3'],
    'LS2_LS1': ['LS3', 'LS2_LS3_050', 'LS2', 'LS1_LS2_025', 'LS1_LS2_050', 'LS1_LS2_075', 'LS1', 'LS1_LPP_050', 'LPP'],
}

# Define relevant targets (6 up + 6 down) for each condition
zone_relevant_targets = {
    'LS1_LPP': {
        'LS1': ['LS3', 'LS2_LS3_050', 'LS2', 'LS1_LS2_025', 'LS1_LS2_050', 'LS1_LS2_075',
                'LS1_LPP_025', 'LS1_LPP_050', 'LS1_LPP_075', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050'],
        'LS1_LPP_050': ['LS2', 'LS1_LS2_050', 'LS1', 'LS1_LPP_025',
                        'LS1_LPP_075', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075', 'LR1'],
        'LPP': ['LS1_LS2_050', 'LS1', 'LS1_LPP_025', 'LS1_LPP_050', 'LS1_LPP_075',
                'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075', 'LR1', 'LR1_LR2_025', 'LR1_LR2_050'],
    },
    'LPP_LR1': {
        'LPP': ['LS2', 'LS1_LS2_050', 'LS1', 'LS1_LPP_025', 'LS1_LPP_050', 'LS1_LPP_075',
                'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075', 'LR1', 'LR1_LR2_025', 'LR1_LR2_050'],
        'LPP_LR1_050': ['LS1', 'LS1_LPP_050', 'LPP', 'LPP_LR1_025',
                        'LPP_LR1_075', 'LR1', 'LR1_LR2_025', 'LR1_LR2_050', 'LR1_LR2_075', 'LR2'],
        'LR1': ['LS1_LPP_050', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075',
                'LR1_LR2_025', 'LR1_LR2_050', 'LR1_LR2_075', 'LR2', 'LR2_LR3_025', 'LR2_LR3_050'],
    },
    'LR1_LR2': {
        'LR1': ['LS1', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050', 'LPP_LR1_075',
                'LR1_LR2_025', 'LR1_LR2_050', 'LR1_LR2_075', 'LR2', 'LR2_LR3_025', 'LR2_LR3_050'],
        'LR1_LR2_050': ['LPP', 'LPP_LR1_050', 'LR1', 'LR1_LR2_025',
                        'LR1_LR2_075', 'LR2', 'LR2_LR3_025', 'LR2_LR3_050', 'LR2_LR3_075', 'LR3'],
    },
    'LS2_LS1': {
        'LS2': ['LS3', 'LS2_LS3_025', 'LS2_LS3_050', 'LS2_LS3_075',
                'LS1_LS2_025', 'LS1_LS2_050', 'LS1_LS2_075', 'LS1', 'LS1_LPP_025', 'LS1_LPP_050'],
        'LS1': ['LS3', 'LS2_LS3_050', 'LS2', 'LS1_LS2_025', 'LS1_LS2_050', 'LS1_LS2_075',
                'LS1_LPP_025', 'LS1_LPP_050', 'LS1_LPP_075', 'LPP', 'LPP_LR1_025', 'LPP_LR1_050'],
    },
}

all_results = {}

for zone in ['LS1_LPP', 'LPP_LR1', 'LR1_LR2', 'LS2_LS1']:
    if zone not in df_daily['opening_zone'].values:
        print(f'\n[SKIP] Zone {zone}: no days found')
        continue
    
    zone_data = df_daily[df_daily['opening_zone'] == zone]
    n_zone = len(zone_data)
    
    if n_zone < 15:
        print(f'\n[SKIP] Zone {zone}: only {n_zone} days (need >= 15)')
        continue
    
    print(f'\n{"="*90}')
    print(f'ZONE: {zone} (N = {n_zone} days)')
    print(f'{"="*90}')
    
    all_results[zone] = {}
    
    for condition in zone_conditions[zone]:
        df_cond = calculate_conditional_probs_local(df_daily, zone, condition)
        
        if df_cond is None:
            continue
        
        n_condition = sum(
            (row['first_touches'].get(condition) is not None) if row['first_touches'] is not None else False
            for _, row in zone_data.iterrows()
        )
        
        if n_condition < 5:
            continue
        
        # Filter to relevant targets (6 up + 6 down)
        if condition in zone_relevant_targets.get(zone, {}):
            relevant = zone_relevant_targets[zone][condition]
            df_cond_filtered = df_cond[df_cond['target'].isin(relevant)].copy()
        else:
            # Fallback: show all interesting targets
            df_cond_filtered = df_cond[(df_cond['prob_conditional'] > 0.05) & (df_cond['prob_conditional'] < 0.95)].copy()
        
        df_cond_filtered = df_cond_filtered.sort_values('prob_unconditional')
        
        all_results[zone][condition] = df_cond_filtered
        
        print(f'\n  CONDITION: Reached {condition} (N = {n_condition} days, {n_condition/n_zone:.0%} of zone)')
        print(f'  {"Target":<18} {"Cond.Prob":>11} {"Uncond.Prob":>12} {"Delta":>10} {"N_seq":>8}')
        print(f'  {"-"*65}')
        
        for _, row in df_cond_filtered.iterrows():
            if row['target'] == condition:
                continue
            
            if 0.05 < row['prob_conditional'] < 0.95:
                print(f"  {row['target']:18} {row['prob_conditional']:>10.0%} {row['prob_unconditional']:>11.0%} {row['prob_delta']:>9.0%} {row['count_sequential']:>8.0f}")

print(f'\n{"="*90}')
print('[OK] Local pivot conditional probability analysis complete')
print('[CONFIRMED] All probabilities based on 10:00-17:30 session, no temporal bias')
print(f'{"="*90}')

## Step 9: Summary and Interpretation

In [ ]:
print('\n[STEP 9] Summary - FDAX Local Pivots')
print('='*80)
print(f'''
DATA SOURCE: FDAX M5 OHLCV (Databento, local CSV)
PERIOD: {df_daily["date"].min()} to {df_daily["date"].max()}
TRADING DAYS: {len(df_daily)}

KEY METHODOLOGY:
1. Local pivots from first hour (9:00-9:55) using standard formulas
2. Opening zone classified at 10:00
3. Conditional probabilities for 10:00-17:30 session only
4. Temporal ordering: target_ts >= condition_ts
5. NO look-ahead bias (pivots established BEFORE analysis window)

COMPARISON NOTES:
- DE40 study used ~1276 days (2021-2025), this has ~{len(df_daily)} days
- FDAX is the actual futures contract, DE40 is CFD-derived
- FDAX has real exchange volume (Databento), DE40 had broker volume
- Smaller sample = wider confidence intervals, treat N < 30 with caution

NEXT STEPS:
- Compare FDAX results with DE40 for consistency
- If patterns match, confirms edge is real (not broker/data artifact)
- If patterns differ, investigate why (volume, execution, data quality)
''')
print('='*80)

## Step 10: Export Summary

In [ ]:
print('\n[STEP 10] Export Summary')
print('='*80)

export_rows = []

for zone, conditions_dict in all_results.items():
    for condition, df_result in conditions_dict.items():
        for _, row in df_result.iterrows():
            export_rows.append({
                'zone': zone,
                'condition': condition,
                'target': row['target'],
                'prob_conditional': row['prob_conditional'],
                'prob_unconditional': row['prob_unconditional'],
                'prob_delta': row['prob_delta'],
                'count_sequential': row['count_sequential'],
            })

if len(export_rows) > 0:
    df_export = pd.DataFrame(export_rows)
    
    print(f'[OK] Summary table created with {len(df_export)} rows')
    print(f'\nTop 20 strongest local pivot relationships (by delta):')
    top_20 = df_export.nlargest(20, 'prob_delta')[['zone', 'condition', 'target', 'prob_conditional', 'prob_delta', 'count_sequential']]
    print(top_20.to_string(index=False))
    
    print(f'\nTop 20 highest conditional probabilities (>70%):')
    high_prob = df_export[df_export['prob_conditional'] > 0.70].nlargest(20, 'prob_conditional')[['zone', 'condition', 'target', 'prob_conditional', 'count_sequential']]
    if len(high_prob) > 0:
        print(high_prob.to_string(index=False))
    else:
        print('  [No targets with >70% conditional probability]')
else:
    print('[WARNING] No results to export')

print('\n[COMPLETE] FDAX Local Pivot Conditional Probability Analysis Finished')
print('[NOTE] Compare with DE40 results to validate cross-instrument consistency')
print('='*80)